# Add environmental features to Kozani parcels

This reusable feature-engineering notebook adds mean DEM elevation and nearest-lake context to the parcel GeoParquet. Each replacement is atomic, and more environmental feature steps can be added here later.

## Setup

In [1]:
from pathlib import Path
import os
import shutil
from common_libraries.io_library import read_data, write_data
from common_libraries.parcel_features import (
    append_nearest_lake_features,
    append_parcel_elevation,
    append_soil_feature,
    create_dem_mosaic,
    find_dem_tiles,
    validate_projected_metric_crs,
)

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\xicor\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Paths and options

In [2]:
WORK_PATH = Path("C:/work_dir/volvi-2023-2024")
DEM_DIR = WORK_PATH / "dem"
DEM_DIR.mkdir(parents=True, exist_ok=True)
MOSAIC_PATH = DEM_DIR / "dem_mosaic.tif"

PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
LAKES_PATH = Path("C:/work_dir/Lakes.gpkg")
LAKES_LAYER = "lakes_qgis"
SOIL_PATH = Path("C:/work_dir/Agro_Soil.gpkg")
SOIL_LAYER = None

EPSG_CODE = 2100  # Greek Grid (metric)
# False uses pixels whose centers fall inside each parcel. Set True for very small parcels.
ALL_TOUCHED = False

assert PARCELS_PATH.is_file(), f"Missing parcel file: {PARCELS_PATH}"
assert LAKES_PATH.is_file(), f"Missing lake file: {LAKES_PATH}"
assert SOIL_PATH.is_file(), f"Missing soil file: {SOIL_PATH}"

# Load the parcel dataset
parcel_geometry = read_data(str(PARCELS_PATH))
print(f"Loaded parcel dataset with {len(parcel_geometry)} features")
print(f"Original CRS: {parcel_geometry.crs}")

# Ensure the CRS is set correctly
if parcel_geometry.crs is None:
    print(f"Parcel dataset has no CRS; setting to EPSG:{EPSG_CODE}")
    parcel_geometry = parcel_geometry.set_crs(epsg=EPSG_CODE)
    write_data(parcel_geometry, str(PARCELS_PATH))
elif parcel_geometry.crs.to_epsg() != EPSG_CODE:
    print(f"Parcel CRS is {parcel_geometry.crs.to_epsg()}; converting to EPSG:{EPSG_CODE}")
    parcel_geometry = parcel_geometry.to_crs(epsg=EPSG_CODE)
    write_data(parcel_geometry, str(PARCELS_PATH))
else:
    print(f"Parcel dataset already uses EPSG:{EPSG_CODE}")

# Validate the CRS
validate_projected_metric_crs(parcel_geometry.crs, dataset_name="Parcel dataset")
ORIGINAL_PARCEL_CRS = parcel_geometry.crs
print(f"Parcel CRS that will be preserved: {ORIGINAL_PARCEL_CRS}")
parcel_geometry.head()

Reading file: C:/work_dir/volvi-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet
Loaded parcel dataset with 28699 features
Original CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "WGS 84 / UTM zone 35N", "base_crs": {"name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geo

parcel_code  initial_label_code  initial_label  predicted_label_code  \
0   277666994                 119  grass_fodders                  129    
1   277666938                 119  grass_fodders                  111    
2   277666942                 119  grass_fodders                  111    
3   277667008                 110          wheat                  110    
4   277666998                 119  grass_fodders                  110    

  predicted_label  prediction_probability set_type             label_gr  \
0          fallow         0.3141156000       train  χορτοδοτικές καλ...   
1          barley         0.2663752700        test  χορτοδοτικές καλ...   
2          barley         0.1837388100        test  χορτοδοτικές καλ...   
3           wheat         0.4703514000       train               σιτάρι   
4           wheat         0.6944472000        test  χορτοδοτικές καλ...   

              geometry  batch_number  intersected_pixel_count  \
0  POLYGON ((422211...             1                   70       
1  POLYGON ((422390...             1                  332       
2  POLYGON ((422372...             1                  112       
3  POLYGON ((422573...             1                   82       
4  POLYGON ((417378...             1                  117       

   meets_minimum_pixel_count  expected_pixel_count  \
0                 True                      12600    
1                 True                      59760    
2                 True                      20160    
3                 True                      14760    
4                 True                      21060    

   temporal_filled_pixel_count  spatial_filled_pixel_count  \
0                    0                            0          
1                   98                            0          
2                    0                            0          
3                    0                            0          
4                    0                            0          

   temporal_filled_ratio  spatial_filled_ratio  data_reliability_score  \
0         0.0000000000           0.0000000000          1.0000000000      
1         0.0016398929           0.0000000000          0.9983601071      
2         0.0000000000           0.0000000000          1.0000000000      
3         0.0000000000           0.0000000000          1.0000000000      
4         0.0000000000           0.0000000000          1.0000000000      

        pixel_area        geom_area  geom_interior_area_ratio  \
0  7000.0000000000  4981.3191093498         1.4052502653        
1 33200.0000000000 28418.2862207056         1.1682618629        
2 11200.0000000000  7917.8669660429         1.4145223768        
3  8200.0000000000  5964.7819932065         1.3747359098        
4 11700.0000000000  7055.0273609518         1.6583918674        

   geom_compactness  geom_perimeter_area_ratio  geom_shape_index  \
0      0.7869139712         0.0566199031            1.1272918625   
1      0.7158628416         0.0248537016            1.1819118582   
2      0.4742042982         0.0578519615            1.4521693192   
3      0.7561076899         0.0527856431            1.1500273595   
4      0.1787476711         0.0998240957            2.3652649920   

   geom_elongation  geom_shape_complexity_score  \
0     1.0904588833         0.1822107292           
1     1.4410145677         0.2293326098           
2     2.4873750752         0.5436928065           
3     1.0384748149         0.1652668309           
4     2.6529932740         0.7808089937           

   B02_coefficient_of_variation__20231001  \
0         0.2006631463                      
1         0.3632557737                      
2         0.2277948617                      
3         0.0921620253                      
4         0.1095457432                      

   B02_coefficient_of_variation__20231101  \
0         0.2901157921                      
1         0.0691940461                      
2         0.0718721455                      
3         0.07

## Create or reuse the DEM mosaic

In [ ]:
# def_files_path = "F:/Dem/ESA_10"
# gdf_dem_tiles_path = os.path.join(def_files_path, "esa_tiles_dem_10m.gpkg")
# gdf_dem_tiles = read_data(gdf_dem_tiles_path)
# gdf_dem_tiles = gdf_dem_tiles.to_crs(epsg=EPSG_CODE)
# tiles_to_copy = gdf_dem_tiles[gdf_dem_tiles.geometry.intersects(parcel_geometry.union_all())]
# display(tiles_to_copy)
# for _, tile in tiles_to_copy.iterrows():
#     print(f"Processing tile: {tile}")
#     src_tile_path = os.path.join(def_files_path, tile.file_name)
#     dst_tile_path = os.path.join(DEM_DIR, tile.file_name)
#     if not os.path.isfile(dst_tile_path):
#         print(f"Copying {src_tile_path} to {dst_tile_path}")
#         shutil.copy(src_tile_path, dst_tile_path)

In [3]:
dem_tiles = find_dem_tiles(DEM_DIR, MOSAIC_PATH)
# Reuses a valid existing mosaic. Set overwrite=True to rebuild it.
MOSAIC_PATH = create_dem_mosaic(dem_tiles, MOSAIC_PATH, overwrite=False)

Using existing DEM mosaic: C:\work_dir\volvi-2023-2024\dem\dem_mosaic.tif


## Add elevation and overwrite the parcel file

In [4]:
parcels_with_elevation = append_parcel_elevation(
    parcels_path=PARCELS_PATH,
    mosaic_path=MOSAIC_PATH,
    output_path=PARCELS_PATH,
    all_touched=ALL_TOUCHED,
    overwrite=True,
)
assert parcels_with_elevation.crs == ORIGINAL_PARCEL_CRS

Reading file: C:/work_dir/volvi-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet
Parcel dataset CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "GGRS87 / Greek Grid", "base_crs": {"name": "GGRS87", "datum": {"type": "GeodeticReferenceFrame", "name": "Greek Geodetic Reference System 1987", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4121}}, "conversion": {"name": "Greek Grid", "method": {"name": "Transverse Mercator", "id": {"authority": "EPSG", "code": 9807}}, "parameters": [{"name": "Latitude of natural origin", "value": 0, "unit": "degree", "id": {"authority": "EPSG", "code": 8801}

## Add nearest-lake features

Distance is the minimum polygon-to-polygon distance in EPSG:2100 metres. `nearest_lake_row_id` stores the stable GeoPackage feature ID (`fid`) of the matched lake.

In [5]:
parcels_with_features = append_nearest_lake_features(
    parcels_path=PARCELS_PATH,
    lakes_path=LAKES_PATH,
    output_path=PARCELS_PATH,
    lakes_layer=LAKES_LAYER,
    overwrite=True,
)
assert parcels_with_features.crs == ORIGINAL_PARCEL_CRS

Reading file: C:/work_dir/volvi-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet
Reading file: C:/work_dir/Lakes.gpkg Layer: lakes_qgis
Writing file: C:/work_dir/volvi-2023-2024/1_parcel_stats/.satellite_parcel_time_stats.tmp.geoparquet
Saved parcels with nearest-lake features: C:\work_dir\volvi-2023-2024\1_parcel_stats\satellite_parcel_time_stats.geoparquet


## Add the soil attribute

Each parcel receives `GAIOIKANOT` from the soil polygon intersecting its centroid. Centroids are calculated in the original projected parcel CRS.

In [6]:
parcels_with_features = append_soil_feature(
    parcels_path=PARCELS_PATH,
    soil_path=SOIL_PATH,
    output_path=PARCELS_PATH,
    soil_layer=SOIL_LAYER,
    attribute_column="GAIOIKANOT",
    overwrite=True,
)
assert parcels_with_features.crs == ORIGINAL_PARCEL_CRS

Reading file: C:/work_dir/volvi-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet
Reading file: C:/work_dir/Agro_Soil.gpkg Layer: ____
Writing file: C:/work_dir/volvi-2023-2024/1_parcel_stats/.satellite_parcel_time_stats.tmp.geoparquet
Saved parcels with soil feature 'GAIOIKANOT': C:\work_dir\volvi-2023-2024\1_parcel_stats\satellite_parcel_time_stats.geoparquet
Parcels whose centroid did not intersect a soil polygon: 0/28,699


## Checks

In [7]:
feature_columns = [
    "elevation_mean_m",
    "elevation_valid_pixel_count",
    "distance_to_nearest_lake_m",
    "nearest_lake_row_id",
    "GAIOIKANOT",
]
saved_parcels = read_data(str(PARCELS_PATH))
assert saved_parcels.crs == ORIGINAL_PARCEL_CRS, (saved_parcels.crs, ORIGINAL_PARCEL_CRS)
validate_projected_metric_crs(saved_parcels.crs, dataset_name="Saved parcel dataset")
summary = saved_parcels[["elevation_mean_m", "elevation_valid_pixel_count", "distance_to_nearest_lake_m"]].describe()
missing_elevation = int(saved_parcels["elevation_mean_m"].isna().sum())
missing_lake = int(saved_parcels["distance_to_nearest_lake_m"].isna().sum())
missing_soil = int(saved_parcels["GAIOIKANOT"].isna().sum())
print(f"Rows: {len(saved_parcels):,}")
print(f"Preserved projected CRS: {saved_parcels.crs}")
print(f"Missing elevation: {missing_elevation:,}")
print(f"Missing nearest lake: {missing_lake:,}")
print(f"Missing soil attribute: {missing_soil:,}")
display(saved_parcels["GAIOIKANOT"].value_counts(dropna=False).head(10))
summary

Reading file: C:/work_dir/volvi-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet
Rows: 28,699
Preserved projected CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "GGRS87 / Greek Grid", "base_crs": {"name": "GGRS87", "datum": {"type": "GeodeticReferenceFrame", "name": "Greek Geodetic Reference System 1987", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4121}}, "conversion": {"name": "Greek Grid", "method": {"name": "Transverse Mercator", "id": {"authority": "EPSG", "code": 9807}}, "parameters": [{"name": "Latitude of natural origin", "value": 0, "unit": "degree", "id": {"authority": "EP

GAIOIKANOT
A8-111-4-D9EE      11099
T7-111-2-D9QQ       5872
T7-111-1-D9BB       1409
T7-121-1-D9EN       1391
A7-111-1-D9EE       1381
T7-221-1-D9EN       1188
T7-111-1-D9NN       1025
T7T3-121-1-D9NN     1000
X7X3-221-1-D9BB      927
B8A8-771-2-D4EE      312
Name: count, dtype: int64

,elevation_mean_m,elevation_valid_pixel_count,distance_to_nearest_lake_m
count,28699.0000000000,28699.0000000000,28699.0000000000
mean,142.9824670179,62.1190982264,6925.5555504542
std,80.8448544472,48.9172681083,4945.4571449699
min,36.0620073591,15.0000000000,0.0000000000
25%,86.9012568781,35.0000000000,2354.8396516067
50%,113.3735331874,50.0000000000,6049.1007563112
75%,190.5933425819,75.0000000000,11189.3877885827
max,610.1514601935,2283.0000000000,19823.3975368023


In [8]:
saved_parcels.head()

parcel_code  initial_label_code  initial_label  predicted_label_code  \
0   277666994                 119  grass_fodders                  129    
1   277666938                 119  grass_fodders                  111    
2   277666942                 119  grass_fodders                  111    
3   277667008                 110          wheat                  110    
4   277666998                 119  grass_fodders                  110    

  predicted_label  prediction_probability set_type             label_gr  \
0          fallow         0.3141156000       train  χορτοδοτικές καλ...   
1          barley         0.2663752700        test  χορτοδοτικές καλ...   
2          barley         0.1837388100        test  χορτοδοτικές καλ...   
3           wheat         0.4703514000       train               σιτάρι   
4           wheat         0.6944472000        test  χορτοδοτικές καλ...   

              geometry  batch_number  intersected_pixel_count  \
0  POLYGON ((422211...             1                   70       
1  POLYGON ((422390...             1                  332       
2  POLYGON ((422372...             1                  112       
3  POLYGON ((422573...             1                   82       
4  POLYGON ((417378...             1                  117       

   meets_minimum_pixel_count  expected_pixel_count  \
0                 True                      12600    
1                 True                      59760    
2                 True                      20160    
3                 True                      14760    
4                 True                      21060    

   temporal_filled_pixel_count  spatial_filled_pixel_count  \
0                    0                            0          
1                   98                            0          
2                    0                            0          
3                    0                            0          
4                    0                            0          

   temporal_filled_ratio  spatial_filled_ratio  data_reliability_score  \
0         0.0000000000           0.0000000000          1.0000000000      
1         0.0016398929           0.0000000000          0.9983601071      
2         0.0000000000           0.0000000000          1.0000000000      
3         0.0000000000           0.0000000000          1.0000000000      
4         0.0000000000           0.0000000000          1.0000000000      

        pixel_area        geom_area  geom_interior_area_ratio  \
0  7000.0000000000  4981.3191093498         1.4052502653        
1 33200.0000000000 28418.2862207056         1.1682618629        
2 11200.0000000000  7917.8669660429         1.4145223768        
3  8200.0000000000  5964.7819932065         1.3747359098        
4 11700.0000000000  7055.0273609518         1.6583918674        

   geom_compactness  geom_perimeter_area_ratio  geom_shape_index  \
0      0.7869139712         0.0566199031            1.1272918625   
1      0.7158628416         0.0248537016            1.1819118582   
2      0.4742042982         0.0578519615            1.4521693192   
3      0.7561076899         0.0527856431            1.1500273595   
4      0.1787476711         0.0998240957            2.3652649920   

   geom_elongation  geom_shape_complexity_score  \
0     1.0904588833         0.1822107292           
1     1.4410145677         0.2293326098           
2     2.4873750752         0.5436928065           
3     1.0384748149         0.1652668309           
4     2.6529932740         0.7808089937           

   B02_coefficient_of_variation__20231001  \
0         0.2006631463                      
1         0.3632557737                      
2         0.2277948617                      
3         0.0921620253                      
4         0.1095457432                      

   B02_coefficient_of_variation__20231101  \
0         0.2901157921                      
1         0.0691940461                      
2         0.0718721455                      
3         0.07